# S&P500/VIX Discrete Latent

Main promoted public: standard causal VQ tokenizer plus additive scalar-conditioned causal AR prior.


## Setup


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Markdown, display

AUTO_SELECT_MODEL = True
MODEL_REGISTRY_PATH = "trained_models/model_registry.yaml"
MODEL_SELECTION_PROFILE = "balanced_market"
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_HEAVY = False

RUN_LIGHT = True
RUN_HEAVY = False
USE_RESEARCH_VARIANT = False


RUN_TRAINING = False
RUN_EVALUATION = False


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_yaml(path: str | Path) -> dict[str, Any]:
    resolved = repo_path(path)
    if not resolved.exists():
        return {}
    loaded = yaml.safe_load(resolved.read_text())
    return loaded if isinstance(loaded, dict) else {}


def load_json(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


def print_command(command: list[str]) -> None:
    print(" ".join(shlex.quote(part) for part in command))


def maybe_run(command: list[str], *, enabled: bool, label: str) -> None:
    print(f"{label} command:")
    print_command(command)
    if enabled:
        subprocess.run(command, cwd=REPO_ROOT, check=True)
    else:
        print(f"{label} skipped; set RUN_HEAVY=True and the matching run flag to execute it.")


print(f"Repository root: {REPO_ROOT}")
EXPERIMENT_ID = "sp500_vix"
TOKENIZER_CONFIG = "configs/experiments/sp500_vix_causal_vq_tokenizer.yaml"
TOKEN_PRIOR_CONFIG = "configs/experiments/sp500_vix_causal_token_prior_additive.yaml"
TOKENIZER_DIR = "outputs/sp500_vix_discrete/tokenizer/sp500_vix_causal_vq_tokenizer_seed0"
TOKEN_DATA_DIR = "outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16"
TOKEN_PRIOR_DIR = "outputs/sp500_vix_discrete/token_prior/additive/<prior-run>"
EVALUATION_DIR = "outputs/sp500_vix_discrete/paper_style"
GEOMETRY_DIR = "outputs/latent_geometry/sp500_vix_standard_vq"
BASE_DATA_DIR = "data/processed"

RESEARCH_TOKENIZER_CONFIG = "configs/experiments/sp500_vix_causal_vq_tokenizer_hidden128.yaml"
RESEARCH_TOKEN_PRIOR_CONFIG = "configs/experiments/sp500_vix_causal_token_prior_hidden128_conv_transformer.yaml"
RESEARCH_OUTPUT_DIR = "outputs/sp500_vix_discrete/best_discrete_research/paper_style"

print("Default workflow: standard VQ tokenizer + additive scalar-conditioned causal AR prior.")
print("RVQ q2 was evaluated on research branches and is not part of the public baseline.")


## Registered Model Selection


In [ ]:
from time_causal_vae.experiments.model_registry import load_registry, select_registered_model

REGISTERED_MODEL = None
DISCRETE_CHECKPOINT_CONVENTION = None
if AUTO_SELECT_MODEL:
    registry = load_registry(repo_path(MODEL_REGISTRY_PATH))
    REGISTERED_MODEL = select_registered_model(
        registry,
        experiment=EXPERIMENT_ID,
        family="discrete",
        profile=MODEL_SELECTION_PROFILE,
    )
    if REGISTERED_MODEL.tokenizer_config:
        TOKENIZER_CONFIG = REGISTERED_MODEL.tokenizer_config
    if REGISTERED_MODEL.prior_config:
        TOKEN_PRIOR_CONFIG = REGISTERED_MODEL.prior_config
    DISCRETE_CHECKPOINT_CONVENTION = REGISTERED_MODEL.checkpoint_paths.get(
        "checkpoint_convention"
    )
    print(
        f"Registered discrete model: {REGISTERED_MODEL.candidate_id} "
        f"({REGISTERED_MODEL.selected_by})"
    )
    print(f"Tokenizer config: {display_path(TOKENIZER_CONFIG)}")
    print(f"Prior config: {display_path(TOKEN_PRIOR_CONFIG)}")
    if DISCRETE_CHECKPOINT_CONVENTION:
        print(f"Checkpoint convention: {DISCRETE_CHECKPOINT_CONVENTION}")
    if not any(value is True for value in REGISTERED_MODEL.local_checkpoint_status.values()):
        print(
            "No local tokenizer/prior checkpoint was resolved by the registry. Keep RUN_HEAVY, "
            "RUN_TRAINING, and RUN_EVALUATION disabled until local artefacts are available."
        )
    if REGISTERED_MODEL.metrics:
        display(pd.DataFrame([REGISTERED_MODEL.metrics]))
    if REGISTERED_MODEL.missing_metrics:
        print("Missing metrics:", ", ".join(REGISTERED_MODEL.missing_metrics))
else:
    print("AUTO_SELECT_MODEL=False; using notebook-local config defaults.")


## Configurations


## Optional Research Variant

The hidden128 VQ tokenizer with a causal conv-transformer k3 prior is optional. It is shown only when `USE_RESEARCH_VARIANT=True` and the local research configs or output summaries exist. Missing paths are reported without failing the notebook.

In [ ]:
research_paths = {
    "tokenizer_config": Path(RESEARCH_TOKENIZER_CONFIG),
    "token_prior_config": Path(RESEARCH_TOKEN_PRIOR_CONFIG),
    "paper_style_summary": Path(RESEARCH_OUTPUT_DIR) / "paper_style_summary.json",
}
research_rows = []
for name, path in research_paths.items():
    research_rows.append({"item": name, "path": display_path(path), "exists": repo_path(path).exists()})
display(pd.DataFrame(research_rows))

if not USE_RESEARCH_VARIANT:
    print("USE_RESEARCH_VARIANT=False; staying on the public standard-VQ + additive-AR workflow.")
else:
    summary = load_json(Path(RESEARCH_OUTPUT_DIR) / "paper_style_summary.json")
    if summary is None:
        print(
            "Research variant requested, but no local paper_style_summary.json was found at "
            f"{display_path(Path(RESEARCH_OUTPUT_DIR) / 'paper_style_summary.json')}"
        )
    else:
        comparisons = summary.get("comparisons", {})
        discrete_metrics = comparisons.get("discrete", {}) if isinstance(comparisons, dict) else {}
        display(pd.DataFrame([discrete_metrics]) if discrete_metrics else pd.DataFrame([{"status": "summary loaded"}]))

In [ ]:
for name, config_path in [("tokenizer", TOKENIZER_CONFIG), ("token prior", TOKEN_PRIOR_CONFIG)]:
    if config_path is None:
        print(f"No {name} config is available for this benchmark.")
        continue
    display(Markdown(f"### {name}: `{display_path(config_path)}`"))
    raw_config = load_yaml(config_path)
    rows = []
    for section, value in raw_config.items():
        if isinstance(value, dict):
            for key, item in value.items():
                if isinstance(item, (str, int, float, bool)) or item is None:
                    rows.append({"section": section, "field": key, "value": item})
    display(pd.DataFrame(rows) if rows else pd.DataFrame([{"status": "empty config"}]))

## Pipeline Commands

Commands are printed by default. `RUN_LIGHT=True` keeps the notebook suitable for lightweight execution. Training, token extraction, latent-geometry generation, and evaluation only run when `RUN_HEAVY=True` and the matching run flag is enabled.

In [ ]:
if TOKENIZER_CONFIG is None:
    print(
        "Discrete pipeline commands are unavailable because this benchmark has no committed discrete configs."
    )
else:
    tokenizer_train_command = [
        "poetry",
        "run",
        "tcvae-train-tokenizer",
        "--config",
        display_path(TOKENIZER_CONFIG),
        "--output-dir",
        display_path(Path(TOKENIZER_DIR).parent),
        "--base-data-dir",
        display_path(BASE_DATA_DIR),
        "--no-wandb",
    ]
    tokenizer_dry_run_command = tokenizer_train_command + ["--dry-run"]
    token_extract_command = [
        "poetry",
        "run",
        "python",
        "scripts/extract_token_indices.py",
        "--config",
        display_path(TOKENIZER_CONFIG),
        "--tokenizer-dir",
        display_path(TOKENIZER_DIR),
        "--output-dir",
        display_path(TOKEN_DATA_DIR),
        "--base-data-dir",
        display_path(BASE_DATA_DIR),
        "--seed",
        "99",
    ]
    geometry_command = [
        "poetry",
        "run",
        "python",
        "scripts/analyze_discrete_latent_geometry.py",
        "--config",
        display_path(TOKENIZER_CONFIG),
        "--tokenizer-dir",
        display_path(TOKENIZER_DIR),
        "--token-data-dir",
        display_path(TOKEN_DATA_DIR),
        "--output-dir",
        display_path(GEOMETRY_DIR),
        "--base-data-dir",
        display_path(BASE_DATA_DIR),
        "--plot-voronoi",
    ]
    print("Tokenizer dry-run command:")
    print_command(tokenizer_dry_run_command)
    maybe_run(tokenizer_train_command, enabled=RUN_HEAVY and RUN_TRAINING, label="Tokenizer training")
    maybe_run(token_extract_command, enabled=RUN_HEAVY and RUN_EVALUATION, label="Token extraction")
    maybe_run(geometry_command, enabled=RUN_HEAVY and RUN_EVALUATION, label="Latent geometry")

if TOKEN_PRIOR_CONFIG is not None:
    prior_dry_run_command = [
        "poetry",
        "run",
        "tcvae-train-token-prior",
        "--config",
        display_path(TOKEN_PRIOR_CONFIG),
        "--output-dir",
        display_path(Path(TOKEN_PRIOR_DIR).parent),
        "--no-wandb",
        "--dry-run",
    ]
    prior_train_command = [part for part in prior_dry_run_command if part != "--dry-run"]
    prior_eval_command = [
        "poetry",
        "run",
        "tcvae-evaluate-token-prior",
        "--config",
        display_path(TOKEN_PRIOR_CONFIG),
        "--prior-dir",
        display_path(TOKEN_PRIOR_DIR),
        "--tokenizer-dir",
        display_path(TOKENIZER_DIR),
        "--output-dir",
        display_path(EVALUATION_DIR),
        "--base-data-dir",
        display_path(BASE_DATA_DIR),
        "--n-sample",
        "256",
        "--seed",
        "99",
        "--temperature",
        "1.0",
    ]
    print()
    print("Token-prior dry-run command:")
    print_command(prior_dry_run_command)
    maybe_run(prior_train_command, enabled=RUN_HEAVY and RUN_TRAINING, label="Token-prior training")
    maybe_run(prior_eval_command, enabled=RUN_HEAVY and RUN_EVALUATION, label="Token-prior evaluation")

## Discrete Metrics

The notebook reads local summaries when they exist. Missing files are reported as paths to generate later.


In [ ]:
metric_sources = {
    "tokenizer_summary": Path(EVALUATION_DIR) / "tokenizer_summary.json",
    "token_prior_summary": Path(EVALUATION_DIR) / "token_prior_summary.json",
    "geometry_summary": Path(GEOMETRY_DIR) / "codebook_geometry_summary.json",
}
for name, path in metric_sources.items():
    payload = load_json(path)
    if payload is None:
        print(f"{name}: missing at {display_path(path)}")
        continue
    display(Markdown(f"### {name}: `{display_path(path)}`"))
    if name == "geometry_summary":
        usage = payload.get("usage", {})
        geometry = payload.get("geometry", {})
        metadata = payload.get("metadata", {})
        counts = usage.get("code_usage_counts") or []
        active_counts = [count for count in counts if count]
        display(
            pd.DataFrame([
                {
                    "quantizer_type": metadata.get("quantizer_type"),
                    "embedding_shape": geometry.get("embedding_shape"),
                    "active_codes": usage.get("active_code_count"),
                    "active_code_ratio": usage.get("active_code_ratio"),
                    "perplexity": usage.get("codebook_perplexity"),
                    "token_count": usage.get("token_count"),
                    "token_usage_nonzero_counts": active_counts[:12],
                    "entropy": usage.get("entropy") or usage.get("index_entropy"),
                }
            ])
        )
    else:
        flattened = {
            key: value
            for key, value in payload.items()
            if isinstance(value, (str, int, float, bool))
        }
        display(
            pd.DataFrame([flattened]) if flattened else pd.DataFrame([{"status": "summary loaded"}])
        )

## Torchview Diagram Helper

Torchview is optional and is not imported by package source. Load or instantiate a tokenizer or token prior, assign it to the placeholder variable, and provide representative input data.


In [ ]:
try:
    from torchview import draw_graph
except ImportError:
    draw_graph = None


def show_model_graph(model: Any, *, input_data: Any | None = None, name: str = "model") -> None:
    if model is None:
        print(f"No {name} instance is loaded. Load or instantiate it first, then rerun this cell.")
        return
    if draw_graph is None:
        print(
            "Optional torchview support is unavailable. Install the notebooks group with `poetry install --with notebooks`."
        )
        return
    if input_data is None:
        print(
            "Provide representative `input_data` for torchview, for example a small token or path batch."
        )
        return
    graph = draw_graph(model, input_data=input_data, expand_nested=True)
    display(graph.visual_graph)


TOKENIZER_MODEL = None
TOKEN_PRIOR_MODEL = None
show_model_graph(TOKENIZER_MODEL, name="tokenizer")
show_model_graph(TOKEN_PRIOR_MODEL, name="token prior")

## Promoted S&P500/VIX Diagnostics

This cell prints the final command. It only runs when `RUN_EVALUATION=True` and the prior/checkpoint placeholders are replaced with real local paths.


In [ ]:
paper_style_command = [
    "poetry",
    "run",
    "python",
    "scripts/evaluate_sp500_vix_paper_style.py",
    "--discrete-config",
    display_path(TOKEN_PRIOR_CONFIG),
    "--discrete-prior-dir",
    display_path(TOKEN_PRIOR_DIR),
    "--discrete-tokenizer-dir",
    display_path(TOKENIZER_DIR),
    "--continuous-config",
    "configs/experiments/sp500_vix_beta_cvae.yaml",
    "--continuous-model-dir",
    "<continuous-final-model-dir>",
    "--output-dir",
    display_path(EVALUATION_DIR),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--n-sample",
    "1000",
    "--seed",
    "99",
    "--temperature",
    "1.0",
    "--top-k",
    "40",
]
maybe_run(
    paper_style_command,
    enabled=RUN_HEAVY and RUN_EVALUATION and "<" not in TOKEN_PRIOR_DIR,
    label="paper-style evaluation",
)

## Figure Placeholders

Primary report figures are expected under `outputs/sp500_vix_discrete/paper_style/` and `outputs/latent_geometry/sp500_vix_standard_vq/`. The report notebook displays the full figure set.
